# Breast Cancer SVC: Forward Selection with Calibrated Probabilities

This example adapts the breast-cancer forward-selection workflow to `SVC(probability=True)`. MiSTIC uses calibrated positive-class probabilities for ROC scoring and contribution ranking, exposes them through `predict_proba`, and can explain them with integrated gradients.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (ConfusionMatrixDisplay, brier_score_loss,
                             classification_report, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from mistic import combined_rank, cvSet, kernelWrapper, paramSet, score_svc, svmSet

## Load and split the Wisconsin Diagnostic Breast Cancer data

In [ ]:
data_path = Path("wdbc.data")
if not data_path.exists():
    data_path = Path("mistic/examples/wdbc.data")

properties = ["Radius", "Texture", "Perimeter", "Area", "Smoothness",
              "Compactness", "Concavity", "Concave Points", "Symmetry",
              "Fractal Dimension"]
columns = (["ID", "Diagnosis"] + [f"{name} mean" for name in properties]
           + [f"{name} SE" for name in properties]
           + [f"{name} worst" for name in properties])
data = pd.read_csv(data_path, header=None, names=columns)
X = data.drop(columns=["ID", "Diagnosis"])
y = data["Diagnosis"]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=0)
scaler = StandardScaler().fit(X_train_raw)
X_train = scaler.transform(X_train_raw)
X_test = scaler.transform(X_test_raw)
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()
data.shape, np.unique(y_train, return_counts=True)

## Configure probability-enabled SVC and forward selection

In [ ]:
splits = cvSet(X=X_train, y=y_train)
splits.classification(num_sets=3, validation_size=0.20, random_seed=0)

svc = SVC(
    kernel="precomputed",
    class_weight="balanced",
    probability=True,
    random_state=0,
    tol=1e-12,
)
parameter_grid = [
    paramSet(model={"C": cost}, kernel={"gamma": gamma})
    for cost in (0.5, 2.0, 8.0)
    for gamma in (2**-7, 2**-4)
]

# The tuning score assigns 20% to 1-Brier probability calibration; the
# remaining 80% uses an equal ROC-AUC/F1 blend.
probability_scorer = score_svc(weight=0.5, calibration_weight=0.2)
ensemble = svmSet(
    svc, splits, score_method=probability_scorer.score,
    kernel=kernelWrapper(type="rbf"),
    separate_feature_sets=True,
    separate_parameters=True,
)

In [ ]:
ensemble.greedy_forward_selection(
    parameter_grid=parameter_grid,
    addition_factor=0.20,
    max_features=12,
    feature_ranker=combined_rank(weight=0.90).compute,
    set_for_rank="sample",
    tune_models_each_step=True,
)

selected_features = X.columns[ensemble.unified_prediction_features_]
selected_features.tolist()

## Selection curve and blind-test probability estimates

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, metric in zip(axes, ("auc", "f1", "calibration", "score")):
    plt.sca(ax)
    ensemble.plot_performance(metric=metric)
    ax.set_title(metric.upper())
fig.tight_layout()

In [ ]:
probability = ensemble.predict_proba(X_test)
positive_class = ensemble.unified_model_.classes_[1]
positive_probability = probability[:, 1]
prediction = ensemble.predict(X_test)

print(f"Positive class: {positive_class}")
print(f"Blind-test ROC AUC: {roc_auc_score(y_test, positive_probability):.3f}")
print(f"Blind-test Brier loss: {brier_score_loss(y_test, positive_probability, pos_label=positive_class):.3f}")
print(classification_report(y_test, prediction))
ConfusionMatrixDisplay.from_predictions(y_test, prediction)

In [ ]:
# Row-level blind predictions from the final unified model. Probability
# columns follow unified_model_.classes_, which is [B, M] for this data.
member_set_probability = ensemble.predict_proba(
    X_test, prediction_mode="set")
blind_predictions = pd.DataFrame({
    "sample_id": data.loc[X_test_raw.index, "ID"].to_numpy(),
    "observed": y_test,
    "predicted": prediction,
    f"P({ensemble.unified_model_.classes_[0]})": probability[:, 0],
    f"P({ensemble.unified_model_.classes_[1]})": probability[:, 1],
    "member_set_positive_probability": member_set_probability[:, 1],
})
blind_predictions["correct"] = (
    blind_predictions["observed"] == blind_predictions["predicted"])
blind_predictions.to_csv("BreastCancer_forward_probability_blind_predictions.csv",
                         index=False)
blind_predictions

## Integrated gradients of positive-class probability

In [ ]:
# Probability IG currently explains the mean member-set probability.
# A zero vector is a natural reference because the inputs are standardized.
reference = np.zeros(X_test.shape[1])
ig_probability = ensemble.explain_integrated_gradients(
    X_test,
    feature_names=X.columns,
    target=y_test,
    reference_point=reference,
    num_steps=100,
    output="probability",
)

fig, ax = plt.subplots(figsize=(10, 7))
ig_probability.summary_plot(
    ax=ax, max_features=15, cmap="coolwarm", random_state=0,
    scatter_kwargs={"s": 26, "alpha": 0.7, "edgecolors": "none"},
)
ax.set_title(f"Integrated gradients of P({positive_class})")
ax.set_xlabel("Contribution to positive-class probability")
plt.tight_layout()

In [ ]:
# Completeness check: attribution sums approximate the change in the
# mean member-set probability relative to the reference.
reference_rows = np.broadcast_to(reference, X_test.shape)
expected_change = (
    ensemble.predict_proba(X_test, prediction_mode="set")[:, 1]
    - ensemble.predict_proba(reference_rows, prediction_mode="set")[:, 1]
)
attributed_change = ig_probability.values.sum(axis=1)
pd.DataFrame({
    "expected_probability_change": expected_change,
    "attributed_change": attributed_change,
    "residual": expected_change - attributed_change,
}).describe()